# Flight Delay Prediction API

This project started as part of my graduate work in data analytics, but it ended up teaching me a lot about what happens after you build a machine learning model.

It’s one thing to train a model in a notebook. It’s another thing entirely to make it usable.

The goal of this project was to build an API that could take flight information as input and return a predicted departure delay in real time. In other words, instead of a model living quietly inside a notebook, this project turns it into something another application or user could actually interact with.

## Project Goal

Most machine learning projects stop after the model is built. I wanted to better understand what happens after that point.

The goal of this project was to take a trained machine learning model and make it usable through an API. Instead of only working inside a notebook, the model can now accept information from a user or another application and return a prediction in real time.

Working on this project helped me better understand how machine learning models connect to real applications, along with all the extra pieces involved like formatting data correctly, validating inputs, testing endpoints, and troubleshooting deployment issues.

## Skills Demonstrated

This project pulled together several different parts of the data workflow instead of focusing on only one skill.

Some of the main skills involved were:

- Training and saving a machine learning model
- Cleaning and formatting incoming data
- Building a FastAPI application
- Creating API endpoints for predictions
- Validating and troubleshooting user input
- Tracking experiments with MLflow
- Testing API functionality
- Organizing project files and dependencies in GitHub

One of the biggest things I learned from this project was how many moving pieces are involved once a model leaves the notebook stage and becomes something another system can actually use.

## Repository Structure

This project is split into separate files based on their purpose so the API, model training, testing, and documentation are easier to manage.

Main files included in the project:

- `main.py` contains the FastAPI application and prediction endpoint
- `model.pkl` stores the trained machine learning model
- `mlflow_experiment.py` handles model training and experiment tracking
- `test_main.py` contains API tests
- `requirements.txt` lists the project dependencies
- `README.md` provides setup instructions and a project overview

Keeping the project organized this way made it easier to test individual pieces, troubleshoot issues, and separate the deployment workflow from the model training process.

## How the API Works

An API (Application Programming Interface) is a way for different applications or systems to communicate with each other.

In this project, the API acts like a middle layer between the user and the trained model.

Instead of opening the model file directly, a user sends flight information to the API. The API checks the information, formats it the way the model expects, sends it to the model, and then returns the predicted delay.

In this project, the basic flow is:

1. The user sends flight details to the prediction endpoint
2. The API checks that the airport codes and time values are usable
3. The flight details are converted into the same format used during training
4. The saved model makes a prediction
5. The API returns the predicted departure delay in minutes

This matters because most people are not going to open a notebook just to use a model. An API gives the model a cleaner way to connect with another application, dashboard, or system.

## Importing the Required Libraries

This project uses a combination of libraries for working with data, loading the saved model, and building the API itself.

Some libraries handle the machine learning side of the project, while others handle the API and prediction workflow.

In [1]:
# Core libraries used in the project

import pandas as pd
import pickle

## Model Features

The model was trained using a few pieces of flight scheduling information:

- Scheduled departure time
- Scheduled arrival time
- Destination airport

The value being predicted was the departure delay in minutes.

This was not meant to be a production airline scheduling system or a perfect prediction engine. The goal was to better understand the full deployment workflow, including how data is prepared, how models receive inputs, and how predictions can be returned through an API.

## What I Learned From This Project

Before working on this project, most of my machine learning experience involved building and testing models inside notebooks.

This project pushed me to think more about everything surrounding the model itself:
- validating user input
- formatting data correctly
- organizing project files
- troubleshooting deployment issues
- testing API behavior
- making the project usable outside a notebook

One of the biggest takeaways for me was realizing that deployment projects involve a lot more problem-solving and debugging than I originally expected.

## Loading the Trained Model

Once the model was trained, it needed to be saved so the API could use it later without retraining everything each time the application started.

This project uses Python's `pickle` library to load the saved model file (`model.pkl`) into the FastAPI application.

In [3]:
import pickle

# Load the trained model
with open("model.pkl", "rb") as model_file:
    model = pickle.load(model_file)

## Converting Time Inputs

The API accepts time values in `HH:MM` format, like `12:30`.

Before the model can use those values, the times need to be converted into numeric values that can be processed mathematically.

For example:

- `12:30` becomes `12.5`
- `14:15` becomes `14.25`

This step helps standardize the incoming data before generating predictions.

In [4]:
def convert_time_to_decimal(time_str: str) -> float:
    hours, minutes = time_str.split(":")
    return int(hours) + int(minutes) / 60

In [5]:
convert_time_to_decimal("12:30")

12.5

## Formatting Incoming Flight Data

Once the API receives the flight details, the information needs to be organized into the same structure the model was trained on.

This step is important because machine learning models expect data in a very specific format. If the columns or structure do not match the training data, predictions can fail or become unreliable.

In [6]:
# Example flight input

departure_airport = "ATL"
arrival_airport = "LAX"

departure_time = "12:30"
arrival_time = "15:45"

# Convert times into decimal values

departure_time_float = convert_time_to_decimal(departure_time)
arrival_time_float = convert_time_to_decimal(arrival_time)

# Create dataframe for prediction

input_data = pd.DataFrame(
    [[arrival_airport, departure_time_float, arrival_time_float]],
    columns=[
        "DEST_AIRPORT",
        "SCHEDULED_DEPARTURE",
        "SCHEDULED_ARRIVAL"
    ]
)

input_data

,DEST_AIRPORT,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL
0,LAX,12.5,15.75


## Generating a Prediction

Once the input data is formatted correctly, it can be passed into the trained model.

The model returns a predicted departure delay in minutes based on the flight information provided.

In [7]:
# Generate prediction

prediction = model.predict(input_data)

prediction

array([-3.93148781])

## Making the Prediction Easier to Read

The raw model output is returned as an array, which is useful for Python but not very friendly for a person reading the notebook.

This step pulls out the prediction value and formats it as a readable sentence.

In [8]:
predicted_delay = prediction[0]

print(f"Predicted departure delay: {predicted_delay:.2f} minutes")

Predicted departure delay: -3.93 minutes


## Connecting Predictions to the API Endpoint

Inside the FastAPI application, the prediction process happens automatically after the API receives flight information from the user.

The endpoint:
- receives the incoming request
- validates the airport codes and time values
- formats the input data
- sends the data to the model
- returns the prediction as a response

This allows another application or user to request predictions without needing direct access to the notebook or model file.

In [9]:
# Example API-style response

response = {
    "predicted_delay_minutes": round(predicted_delay, 2)
}

response

{'predicted_delay_minutes': np.float64(-3.93)}

## Input Validation

The API should not blindly accept every value a user sends.

Before making a prediction, the application checks whether the airport codes are valid. This helps prevent bad inputs from being passed into the model and gives the user a clearer error message when something is wrong.

In [10]:
# Define valid airport codes for this example

VALID_AIRPORTS = {"JFK", "LAX", "ORD", "ATL", "DFW"}

def validate_airports(departure_airport, arrival_airport):
    if departure_airport not in VALID_AIRPORTS or arrival_airport not in VALID_AIRPORTS:
        return False
    return True

In [11]:
validate_airports("ATL", "LAX")

True

In [12]:
validate_airports("ATL", "ZZZ")

False

## Experiment Tracking with MLflow

This project also uses MLflow to track model experiments and training runs.

MLflow helps keep track of:
- model types
- parameters
- evaluation metrics
- saved model versions

This becomes especially useful when testing multiple models or comparing different training approaches over time.

In [13]:
# Example values logged during training

experiment_details = {
    "model_type": "Polynomial Regression (Ridge)",
    "alpha": 1.0,
    "degree": 2,
    "metric": "RMSE"
}

experiment_details

{'model_type': 'Polynomial Regression (Ridge)',
 'alpha': 1.0,
 'degree': 2,
 'metric': 'RMSE'}

## Testing the API

Testing was important to make sure the API returned predictions correctly and handled invalid inputs in a predictable way.

The project includes test cases for:
- successful prediction requests
- invalid airport codes
- incorrect time formatting
- API response behavior

This helped confirm that the application behaved consistently before deployment.

## Challenges During Development

One of the biggest challenges in this project was making sure all of the pieces worked together correctly.

The project involved:
- model training
- preprocessing
- saving the model
- loading the model into the API
- formatting incoming data correctly
- validating user inputs
- troubleshooting prediction errors

A small mismatch between the training data and the API input structure could cause predictions to fail, so debugging and testing became a major part of the workflow.

## Docker and Environment Management

This project also included Docker support to make the application easier to run consistently across different environments.

Docker allows an application and its dependencies to be packaged together into a container so the project behaves more predictably regardless of the machine it runs on.

For deployment-focused projects, this helps reduce issues caused by:
- missing libraries
- dependency conflicts
- environment differences
- inconsistent local setups

Before working on this project, I had mostly worked inside notebooks and local Python environments, so learning how deployment tools like Docker fit into the workflow was a valuable part of the experience.

## What I Would Improve Next

If I continued expanding this project, some next steps would include:

- Supporting a larger list of airport codes
- Connecting the API to live flight or weather data
- Improving input validation
- Adding Docker-based deployment examples
- Creating a frontend dashboard for interacting with the API
- Comparing additional machine learning models

This project gave me a much better understanding of how deployment projects evolve beyond the initial model itself.

## Final Thoughts

This project helped me better understand the difference between building a model and building something usable.

Before this project, most of my work stayed inside notebooks. Working through deployment, validation, debugging, and API behavior gave me a much better appreciation for everything required to move a machine learning project closer to a real application.

It also reinforced how important communication and organization are in technical projects. Building the model was only one part of the process. Making the project understandable, testable, and usable turned out to be just as important.